# NDF & Forward Curve Analysis — BQuant

This notebook runs entirely inside **Bloomberg BQuant** using native `bql` data fetching.
No `pdblp`, no local Bloomberg Terminal required.

### What this notebook does

| Section | Content |
|---|---|
| 1 | Setup — load fxbt2 from BQuant file system |
| 2 | Verify Bloomberg tickers for your pairs |
| 3 | Fetch today's forward curve |
| 4 | Fetch historical curve data (2 years) |
| 5 | Today's curve snapshot |
| 6 | Historical curve overlay (today vs 1M/3M/6M/1Y ago) |
| 7 | Percentile fan chart — cheap or rich vs history? |
| 8 | Implied yield term structure |
| 9 | Level heatmap over time |
| 10 | Full 4-panel dashboard |
| 11 | Single tenor vs spot (premium/discount tracking) |
| 12 | Multi-pair carry comparison |

### Supported pairs
Any pair whose non-USD currency is in the fxbt2 ticker dictionary.
NDF pairs (KRW, INR, IDR, TWD, PHP, BRL, CLP, COP, CNH) fetch outright tickers directly.
Deliverable pairs (EUR, GBP, JPY, AUD, etc.) fetch forward points and construct outrights.

---


## 1 — Setup

Upload the `fxbt2/` folder to your BQuant home directory, then update
`FXBT2_PATH` below to point to the parent folder containing `fxbt2/`.

If you uploaded to `/home/user/fxbt2/`, set `FXBT2_PATH = '/home/user'`.


In [ ]:
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams["figure.dpi"] = 100

# ── Path to fxbt2 ──────────────────────────────────────────────────────────────
# Update this to wherever you uploaded the fxbt2 folder in BQuant
FXBT2_PATH = "/home/user"   # <-- change this
sys.path.insert(0, FXBT2_PATH)

from fxbt2.curves import (
    CurveBuilder, TENORS, TENOR_DAYS, build_tenor_tickers,
    plot_curve_today, plot_curve_history, plot_percentile_bands,
    plot_implied_yields, plot_curve_heatmap, plot_curve_dashboard,
    plot_tenor_vs_spot,
)

# ── BQL Service ────────────────────────────────────────────────────────────────
import bql
bq = bql.Service()

cb = CurveBuilder(fixing="CMPT")

print("Setup complete")
print(f"Available tenors: {TENORS}")

---
## 2 — Verify Bloomberg Tickers

Before fetching data, confirm what tickers will be requested for each pair.
This is useful to check against what you see in the Bloomberg Terminal.

- **NDF pairs** use outright tickers: `KWN+1M CMPT Curncy`, `IRN+3M CMPT Curncy` etc.
- **Deliverable pairs** use forward point tickers: `EUR1M CMPT Curncy`, `JPY3M CMPT Curncy` etc.


In [ ]:
# ── Configure your pairs here ─────────────────────────────────────────────────
NDF_PAIRS         = ["USDKRW", "USDINR", "USDTWD", "USDPHP", "USDIDR"]
DELIVERABLE_PAIRS = ["EURUSD", "USDJPY", "GBPUSD", "USDMXN", "USDBRL"]
ALL_PAIRS         = NDF_PAIRS + DELIVERABLE_PAIRS

# Show ticker table for first pair of each type
for pair in ["USDKRW", "EURUSD"]:
    tickers = build_tenor_tickers(pair, fixing="CMPT")
    print(f"\n{pair} tickers:")
    for tenor, ticker in tickers.items():
        print(f"  {tenor:4s}  {ticker}")

---
## 3 — Fetch Today's Forward Curve

`fetch_curve_bql()` pulls the latest outright rate at each tenor from Bloomberg.
It uses `bql.data.px_last()` for a single reference date.

Set `PAIR` to whichever pair you want to inspect first.


In [ ]:
# ── Choose a pair ─────────────────────────────────────────────────────────────
PAIR = "USDKRW"

curve_today = cb.fetch_curve_bql(PAIR, bq)

print(f"{PAIR} — Forward Curve (today)")
print(curve_today.round(4).to_string())

In [ ]:
# Spot vs curve summary
spot = curve_today.get("1M", float("nan"))  # 1M as reference
print(f"\n{PAIR} forward premium over spot (approx):")
for tenor, val in curve_today.items():
    if not pd.isna(val):
        print(f"  {tenor:4s}  {val:.4f}")

---
## 4 — Fetch Historical Curve Data

`fetch_history_bql()` fetches daily outright rates for all tenors over a date range.
The result is a DataFrame with columns = `spot, ON, 1W, 2W, 1M, 2M, 3M, 6M, 9M, 1Y, 2Y`.

**Change `START` and `END` to your desired window.**
A 2-year history gives sufficient data for percentile analysis.


In [ ]:
# ── Date range ────────────────────────────────────────────────────────────────
START = "2023-01-01"
END   = "2025-01-01"   # adjust to today or desired end date

# Fetch history for the primary pair
print(f"Fetching {PAIR} curve history {START} → {END} ...")
history = cb.fetch_history_bql(PAIR, bq, start=START, end=END)

print(f"\nShape: {history.shape}")
print(f"Columns: {list(history.columns)}")
print(f"Date range: {history.index[0].date()} → {history.index[-1].date()}")
print(f"\nLast 3 rows:")
history.tail(3).round(4)

In [ ]:
# Quick data quality check — how many NaNs per tenor?
print("Missing values per tenor:")
print(history.isna().sum().to_string())

---
## 5 — Today's Curve Snapshot

Plots the forward curve at the latest available date.

- **Upward sloping** = forward premium (outright > spot) → positive carry for USD buyer
- **Downward sloping** = forward discount (outright < spot) → negative carry for USD buyer
- **Steeper curve** = larger interest rate differential at longer tenors


In [ ]:
# Use the latest row of history as today's snapshot
today_curve = history.drop(columns=["spot"]).iloc[-1].dropna()
plot_curve_today(today_curve, pair=PAIR)

In [ ]:
# Side-by-side: NDF pair vs deliverable pair
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Pair 1 (NDF)
c1 = history.drop(columns=["spot"]).iloc[-1].dropna()
plot_curve_today(c1, pair=PAIR, ax=axes[0])

# Pair 2 — fetch a deliverable pair for comparison
PAIR2 = "EURUSD"
print(f"Fetching {PAIR2} history ...")
history2 = cb.fetch_history_bql(PAIR2, bq, start=START, end=END)
c2 = history2.drop(columns=["spot"]).iloc[-1].dropna()
plot_curve_today(c2, pair=PAIR2, ax=axes[1], color="darkorange")

plt.tight_layout()
plt.show()

---
## 6 — Historical Curve Overlay

Overlays the curve at today, 1M ago, 3M ago, 6M ago, and 1Y ago.

**What shifts in the curve mean:**
- **Parallel shift** = overall spot rate moved, carry structure unchanged
- **Steepening** = longer-tenor carry increasing (rate differential widening)
- **Flattening** = carry compressing across the curve
- **Near-end kink** = short-term funding stress or a specific fixing disruption


In [ ]:
plot_curve_history(history, pair=PAIR)

In [ ]:
# Custom dates — compare around a specific event
# e.g. central bank meetings, crisis dates, year-end
custom_dates = [
    "2023-03-01",
    "2023-06-01",
    "2023-09-01",
    "2024-01-02",
    "2024-06-01",
]

plot_curve_history(
    history,
    pair=PAIR,
    snapshot_dates=custom_dates,
    title=f"{PAIR} — Forward Curve at Selected Dates",
)

---
## 7 — Percentile Fan Chart: Cheap or Rich vs History?

The most actionable chart for trade formulation.

Shaded bands show where outright rates have historically traded:
- **Dark band** = 25th–75th percentile (the "normal" range)
- **Light band** = 10th–90th percentile (the broader historical range)
- **Red line** = today's curve

**Signal interpretation:**
| Today's curve vs bands | What it means |
|---|---|
| Above 90th pct | Outrights historically expensive → forward premium stretched |
| 75th–90th pct | Elevated but not extreme |
| 25th–75th pct | In the middle of normal range — no valuation edge |
| 10th–25th pct | Cheap vs history |
| Below 10th pct | Historically cheap → forward discount compressed |

Combine this with a directional signal (carry, momentum) for a full trade view.


In [ ]:
# Full history as reference
plot_percentile_bands(history, pair=PAIR)

In [ ]:
# 1-year rolling window — how does today compare to just the recent period?
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plot_percentile_bands(history, pair=PAIR, ax=axes[0],
                      title=f"{PAIR} — vs Full History")
plot_percentile_bands(history, pair=PAIR, ax=axes[1], hist_window=252,
                      title=f"{PAIR} — vs Last 1 Year")
plt.tight_layout()
plt.show()

---
## 8 — Implied Yield Term Structure

Converts outright rates to annualised implied yields:

```
implied_yield(tenor) = (outright / spot − 1) × (252 / tenor_days)
```

This expresses the carry at each tenor as an annualised percentage,
making it directly comparable to interest rate quotes.

**What to look for:**
- **Upward sloping** = longer tenors imply more carry (normal for EM)
- **Inverted** = near-term yields higher than long-end → short-term funding stress
- **Shift over time** = central bank policy changes appear here before they move spot


In [ ]:
plot_implied_yields(history, pair=PAIR)

In [ ]:
# Get the implied yield history as a DataFrame
yield_history = cb.implied_yield_curve(history)
print(f"Implied yields (annualised %) — last 5 rows:")
(yield_history * 100).round(3).tail()

In [ ]:
# Plot the 1M implied yield time series — useful for carry signal tracking
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(yield_history.index, yield_history["1M"] * 100, lw=1.5, color="steelblue")
ax.axhline(0, color="black", lw=0.7, linestyle=":")
ax.fill_between(yield_history.index, yield_history["1M"] * 100, 0,
                where=yield_history["1M"] > 0, color="steelblue", alpha=0.25, label="Positive carry")
ax.fill_between(yield_history.index, yield_history["1M"] * 100, 0,
                where=yield_history["1M"] < 0, color="crimson", alpha=0.25, label="Negative carry")
ax.set_title(f"{PAIR} — 1M Implied Yield (% ann.) Over Time", fontweight="bold")
ax.set_ylabel("Implied Yield %")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

---
## 9 — Level Heatmap

Shows how outright levels have evolved across all tenors over the full history.

- X-axis = dates (weekly sampled for readability)
- Y-axis = tenors (shortest at bottom)
- Colour = outright level (green = low, red = high)

**What to look for:**
- A smooth colour gradient from bottom to top = stable, well-behaved curve
- Sudden horizontal colour breaks = repricing event across the whole curve
- Vertical stripes = brief period where a specific tenor dislocated from the rest


In [ ]:
plot_curve_heatmap(history, pair=PAIR)

---
## 10 — 4-Panel Dashboard

All four views in a single figure. Use this as your daily curve monitor.


In [ ]:
plot_curve_dashboard(history, pair=PAIR)

---
## 11 — Single Tenor vs Spot: Premium/Discount Tracking

Track how the 3M forward premium or discount has evolved over time.

- **Blue fill** = forward above spot → positive carry (you earn by being long USD in this pair)
- **Red fill** = forward below spot → negative carry

A **widening premium** signals increasing carry — often leads to crowded positioning
and eventual mean-reversion. A **sudden compression** can signal stress or policy shift.


In [ ]:
plot_tenor_vs_spot(history, pair=PAIR, tenor="3M")

In [ ]:
# Compare premium/discount across multiple tenors
tenors_to_check = ["1M", "3M", "6M", "1Y"]
fig, axes = plt.subplots(2, 2, figsize=(16, 8), sharex=True)
axes = axes.flatten()

for i, tenor in enumerate(tenors_to_check):
    if tenor not in history.columns:
        axes[i].set_visible(False)
        continue
    spread = history[tenor] - history["spot"]
    axes[i].fill_between(spread.index, spread.values, 0,
                         where=spread >= 0, color="steelblue", alpha=0.5, label="Premium")
    axes[i].fill_between(spread.index, spread.values, 0,
                         where=spread < 0, color="crimson", alpha=0.5, label="Discount")
    axes[i].axhline(0, color="black", lw=0.8)
    axes[i].set_title(f"{PAIR} {tenor} Forward Premium/Discount", fontweight="bold")
    axes[i].legend(fontsize=7)
    axes[i].grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

---
## 12 — Multi-Pair Carry Comparison

Fetch 1M implied yields for all pairs and compare them side by side.
This is your **carry screen** — a quick view of which pairs offer the most
carry right now and whether current levels are elevated vs history.

Edit `CARRY_PAIRS` to match the pairs you trade.


In [ ]:
# ── Configure carry screen pairs ──────────────────────────────────────────────
CARRY_PAIRS = ["USDKRW", "USDINR", "USDTWD", "USDPHP", "USDIDR",
               "USDBRL", "USDMXN", "EURUSD", "USDJPY"]

TENORS_TO_FETCH = ["1M", "3M", "6M", "1Y"]   # only fetch what you need

print(f"Fetching 1M outright history for {len(CARRY_PAIRS)} pairs ...")
carry_histories = {}
for pair in CARRY_PAIRS:
    try:
        h = cb.fetch_history_bql(pair, bq, start=START, end=END,
                                  tenors=TENORS_TO_FETCH)
        carry_histories[pair] = h
        print(f"  {pair}: {h.shape[0]} rows, {h.dropna().shape[0]} complete rows")
    except Exception as e:
        print(f"  {pair}: FAILED — {e}")

In [ ]:
# Compute 1M implied yield for each pair
cb2 = CurveBuilder()
yield_1m = {}
for pair, hist in carry_histories.items():
    if "1M" in hist.columns and "spot" in hist.columns:
        try:
            yld = cb2.implied_yield_curve(hist, tenors=["1M"])
            yield_1m[pair] = yld["1M"] * 100
        except Exception:
            pass

yield_df = pd.DataFrame(yield_1m).dropna(how="all")
print(f"Yield DataFrame: {yield_df.shape}")
yield_df.tail(3).round(3)

In [ ]:
# ── Plot: time series + today's cross-section ──────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

colors = plt.cm.tab10.colors
for i, pair in enumerate(yield_df.columns):
    axes[0].plot(yield_df.index, yield_df[pair], lw=1.5,
                 label=pair, color=colors[i % 10])
axes[0].axhline(0, color="black", lw=0.7, linestyle=":")
axes[0].set_title("1M Implied Yield (% ann.) — All Pairs", fontweight="bold")
axes[0].set_ylabel("Implied Yield %")
axes[0].legend(fontsize=7, ncol=3)
axes[0].grid(True, alpha=0.25)

# Latest bar chart
latest = yield_df.iloc[-1].dropna().sort_values(ascending=False)
bar_colors = [colors[list(yield_df.columns).index(p) % 10] for p in latest.index]
axes[1].bar(range(len(latest)), latest.values, color=bar_colors, edgecolor="white")
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_xticks(range(len(latest)))
axes[1].set_xticklabels(latest.index.tolist(), fontsize=9)
axes[1].set_title(
    f"Latest 1M Carry — {yield_df.index[-1].strftime('%d %b %Y')}", fontweight="bold")
axes[1].set_ylabel("1M Implied Yield %")
axes[1].grid(True, alpha=0.25, axis="y")
for i, v in enumerate(latest.values):
    axes[1].text(i, v + (0.05 if v >= 0 else -0.15),
                 f"{v:.2f}%", ha="center", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ── Percentile rank of today's 1M yield for each pair (0 = historically cheap) ──
print("1M Implied Yield Percentile Rank vs Full History:")
print(f"{'Pair':10s}  {'Current %':>10s}  {'Pct Rank':>10s}  {'Signal':>10s}")
print("-" * 45)

for pair in latest.index:
    series = yield_df[pair].dropna()
    current = series.iloc[-1]
    pct_rank = (series < current).mean() * 100
    signal = "HIGH CARRY" if pct_rank > 75 else ("LOW CARRY" if pct_rank < 25 else "neutral")
    print(f"{pair:10s}  {current:>10.3f}%  {pct_rank:>9.1f}%  {signal:>10s}")